
# MỨC 5: RANDOM FOREST – HOLD-OUT

## Mục tiêu

Tiếp tục tối ưu Random Forest bằng cách mở rộng phạm vi tìm kiếm siêu tham số từ 36 lên 90 cấu hình.

Quy trình thực nghiệm:
1. Đọc dữ liệu Diamonds Prices2022.csv.
2. Làm sạch dữ liệu giống mức 2, 3 và 4.
3. Chia Train/Test theo tỷ lệ 80/20.
4. Mã hóa các thuộc tính phân loại.
5. Sử dụng GridSearchCV với 3-Fold trên Train.
6. Đánh giá mô hình tốt nhất trên Test.
7. So sánh kết quả với mức 3 và mức 4 Hold-out.


Đọc dữ liệu và làm schaj

In [1]:

# CELL 2: IMPORT CAC THU VIEN CAN THIET

import time  # Thu vien dung de do thoi gian huan luyen

import numpy as np  # Thu vien ho tro tinh toan so hoc

import pandas as pd  # Thu vien doc va xu ly du lieu dang bang

from pathlib import Path  # Ho tro xu ly duong dan tep

from sklearn.model_selection import train_test_split  # Chia du lieu Train/Test

from sklearn.model_selection import GridSearchCV  # Tim kiem sieu tham so tot nhat

from sklearn.model_selection import KFold  # Chia du lieu thanh cac fold

from sklearn.preprocessing import OrdinalEncoder  # Ma hoa thuoc tinh phan loai

from sklearn.compose import ColumnTransformer  # Ap dung tien xu ly cho tung nhom cot

from sklearn.pipeline import Pipeline  # Ket hop tien xu ly va mo hinh

from sklearn.ensemble import RandomForestRegressor  # Mo hinh Random Forest hoi quy

from sklearn.metrics import mean_squared_error  # Tinh MSE

from sklearn.metrics import mean_absolute_error  # Tinh MAE

from sklearn.metrics import r2_score  # Tinh he so R2

print("Da import cac thu vien thanh cong!")  # Thong bao hoan thanh

# CELL 4: DOC VA KIEM TRA DU LIEU THO

# Duong dan khi notebook chay tu thu muc experiments/random_forest
file_path = Path("../../data/raw/Diamonds Prices2022.csv")

# Neu khong tim thay file thi thu duong dan tu thu muc goc
if not file_path.exists():
    file_path = Path("data/raw/Diamonds Prices2022.csv")

# Neu ca hai duong dan deu khong ton tai thi bao loi
if not file_path.exists():
    raise FileNotFoundError("Khong tim thay file Diamonds Prices2022.csv")

# Doc du lieu CSV va dung cot dau tien lam chi so dong
diamonds = pd.read_csv(file_path, index_col=0)

# In duong dan file dang duoc su dung
print("Duong dan:", file_path.resolve())

# In so dong va so cot cua du lieu
print("Kich thuoc du lieu:", diamonds.shape)

# Dem tong so gia tri bi thieu
print("So gia tri thieu:", diamonds.isnull().sum().sum())

# Dem so dong trung lap trong du lieu
print("So dong trung lap:", diamonds.duplicated().sum())

# Dem so dong co it nhat mot kich thuoc x, y, z khong hop le
print(
    "So dong co kich thuoc <= 0:",
    (
        (diamonds["x"] <= 0) |  # Kiem tra chieu dai x
        (diamonds["y"] <= 0) |  # Kiem tra chieu rong y
        (diamonds["z"] <= 0)    # Kiem tra chieu cao z
    ).sum()  # Dem tong so dong thoa dieu kien
)

# Hien thi 5 dong dau tien
display(diamonds.head())


Da import cac thu vien thanh cong!
Duong dan: D:\DiamondPrediction\data\raw\Diamonds Prices2022.csv
Kich thuoc du lieu: (53943, 10)
So gia tri thieu: 0
So dong trung lap: 149
So dong co kich thuoc <= 0: 20


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Tiền xử lý dữ liệu

In [2]:

# CELL 6: TIEN XU LY DU LIEU

# Tao ban sao du lieu goc de xu ly
df = diamonds.copy()

# Luu so dong ban dau
initial_rows = len(df)

# Xoa cac dong co gia tri thieu
df = df.dropna()

# Luu so dong sau khi xoa gia tri thieu
after_missing = len(df)

# Chi giu nhung dong co kich thuoc x, y, z lon hon 0
df = df[
    (df["x"] > 0) &  # Chieu dai phai duong
    (df["y"] > 0) &  # Chieu rong phai duong
    (df["z"] > 0)    # Chieu cao phai duong
]

# Luu so dong sau khi loc kich thuoc
after_dimensions = len(df)

# Xoa cac dong trung lap
df = df.drop_duplicates()

# Dat lai chi so dong tu 0
df = df.reset_index(drop=True)

# In so dong bi loai do thieu du lieu
print("Dong thieu bi loai:", initial_rows - after_missing)

# In so dong bi loai do kich thuoc khong hop le
print("Dong kich thuoc khong hop le:",
      after_missing - after_dimensions)

# In so dong trung lap bi loai
print("Dong trung lap bi loai:",
      after_dimensions - len(df))

# In so dong con lai sau tien xu ly
print("So dong sau tien xu ly:", len(df))

# In so cot cua du lieu
print("So cot:", df.shape[1])

# Hien thi 5 dong dau sau tien xu ly
display(df.head())


Dong thieu bi loai: 0
Dong kich thuoc khong hop le: 20
Dong trung lap bi loai: 148
So dong sau tien xu ly: 53775
So cot: 10


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Chia dữ liệu Hold-out

In [3]:

# CELL 8: CHIA DU LIEU TRAIN VA TEST

# Lay tat ca thuoc tinh dau vao, bo cot price
X = df.drop(columns=["price"])

# Lay cot price lam gia tri can du doan
y = df["price"]

# Chia du lieu thanh 80% Train va 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X,  # Du lieu dac trung dau vao
    y,  # Gia kim cuong thuc te
    test_size=0.2,  # Su dung 20% du lieu de kiem tra
    random_state=42  # Co dinh cach chia ngau nhien
)

# In tong so mau
print("Tong so mau:", len(X))

# In so mau dung de huan luyen
print("So mau Train:", len(X_train))

# In so mau dung de kiem tra
print("So mau Test:", len(X_test))


Tong so mau: 53775
So mau Train: 43020
So mau Test: 10755


Tạo Pipeline

In [4]:

# CELL 10: XAY DUNG PIPELINE

# Khai bao thu tu chat luong mau sac kim cuong
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Khai bao thu tu do tinh khiet kim cuong
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Khai bao thu tu chat luong giac cat
cut_order = [
    "Fair", "Good", "Very Good",
    "Premium", "Ideal"
]

# Danh sach cac cot co gia tri dang chu
categorical_cols = ["color", "clarity", "cut"]

# Tao bo ma hoa cac thuoc tinh phan loai
encoder = OrdinalEncoder(
    categories=[
        color_order,  # Thu tu ma hoa cot color
        clarity_order,  # Thu tu ma hoa cot clarity
        cut_order  # Thu tu ma hoa cot cut
    ]
)

# Tao bo tien xu ly du lieu
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",  # Ten buoc ma hoa
            encoder,  # Su dung OrdinalEncoder
            categorical_cols  # Cac cot can ma hoa
        )
    ],
    remainder="passthrough"  # Giu nguyen cac cot du lieu so
)

# Khoi tao mo hinh Random Forest hoi quy
rf_model = RandomForestRegressor(
    random_state=42,  # Co dinh qua trinh ngau nhien
    n_jobs=1  # Moi mo hinh su dung mot luong xu ly
)

# Tao Pipeline ket hop tien xu ly va Random Forest
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),  # Buoc ma hoa du lieu
        ("model", rf_model)  # Buoc huan luyen Random Forest
    ]
)

# Thong bao tao Pipeline thanh cong
print("Da tao Pipeline thanh cong!")


Da tao Pipeline thanh cong!


Khai báo 90 cấu hình

In [5]:

# CELL 12: KHAI BAO PHAM VI TIM KIEM SIEU THAM SO

# Khai bao cac gia tri sieu tham so can thu nghiem
param_grid = {
    "model__n_estimators": [100, 150, 200, 250, 300],  # So cay trong rung
    "model__max_depth": [None, 20, 30],  # Do sau toi da cua cay
    "model__min_samples_leaf": [1, 2],  # So mau toi thieu tai nut la
    "model__max_features": [0.6, 0.8, 1.0]  # Ty le dac trung xet tai moi lan chia
}

# Tinh tong so cau hinh sieu tham so
number_of_configs = (
    len(param_grid["model__n_estimators"]) *
    len(param_grid["model__max_depth"]) *
    len(param_grid["model__min_samples_leaf"]) *
    len(param_grid["model__max_features"])
)

# Tao K-Fold voi 3 fold de tim sieu tham so
inner_kf = KFold(
    n_splits=3,  # Chia Train thanh 3 fold
    shuffle=True,  # Tron du lieu truoc khi chia
    random_state=42  # Co dinh cach chia
)

# Tao GridSearchCV de tim mo hinh tot nhat
grid_search = GridSearchCV(
    estimator=pipeline,  # Pipeline can toi uu
    param_grid=param_grid,  # Danh sach 90 cau hinh
    scoring="neg_root_mean_squared_error",  # Danh gia theo RMSE
    cv=inner_kf,  # Su dung 3-Fold Cross-validation
    n_jobs=-1,  # Su dung cac loi CPU kha dung
    refit=True,  # Huan luyen lai cau hinh tot nhat tren toan bo Train
    verbose=1  # Hien thi tien trinh huan luyen
)

# In tong so cau hinh
print("So cau hinh:", number_of_configs)

# In so luot huan luyen trong Cross-validation
print("So luot CV:", number_of_configs * 3)


So cau hinh: 90
So luot CV: 270


Huấn luyện GridSearchCV

In [6]:

# CELL 14: HUAN LUYEN VA TIM SIEU THAM SO TOT NHAT

# Ghi lai thoi diem bat dau huan luyen
start_time = time.perf_counter()

# Huan luyen GridSearchCV tren tap Train
grid_search.fit(X_train, y_train)

# Tinh tong thoi gian huan luyen
training_time = time.perf_counter() - start_time

# Lay Pipeline co bo sieu tham so tot nhat
best_model = grid_search.best_estimator_

# Lay bo sieu tham so tot nhat
best_params = grid_search.best_params_

# Lay RMSE trung binh tot nhat trong Cross-validation
best_cv_rmse = -grid_search.best_score_

# Thong bao da huan luyen xong
print("Da hoan thanh GridSearchCV!")

# In bo sieu tham so tot nhat
print("\nBo sieu tham so tot nhat:")
print(best_params)

# In RMSE Cross-validation tot nhat
print(f"\nRMSE CV tot nhat: {best_cv_rmse:.4f}")

# In thoi gian huan luyen tinh bang giay
print(f"Thoi gian huan luyen: {training_time:.2f} giay")


Fitting 3 folds for each of 90 candidates, totalling 270 fits
Da hoan thanh GridSearchCV!

Bo sieu tham so tot nhat:
{'model__max_depth': 30, 'model__max_features': 0.6, 'model__min_samples_leaf': 1, 'model__n_estimators': 300}

RMSE CV tot nhat: 552.2089
Thoi gian huan luyen: 1532.71 giay


Đánh giá mô hình trên Test

In [7]:

# CELL 16: DANH GIA MO HINH TREN TAP TEST

# Su dung mo hinh tot nhat de du doan gia kim cuong
y_pred = best_model.predict(X_test)

# Tinh sai so binh phuong trung binh
mse = mean_squared_error(y_test, y_pred)

# Tinh can bac hai cua MSE
rmse = np.sqrt(mse)

# Tinh sai so tuyet doi trung binh
mae = mean_absolute_error(y_test, y_pred)

# Tinh he so xac dinh R2
r2 = r2_score(y_test, y_pred)

# In ten thi nghiem
print("MUC 5 - RANDOM FOREST HOLD-OUT")

# In duong phan cach
print("-" * 40)

# In MSE tren tap Test
print(f"MSE Test : {mse:.4f}")

# In RMSE tren tap Test
print(f"RMSE Test: {rmse:.4f}")

# In MAE tren tap Test
print(f"MAE Test : {mae:.4f}")

# In he so R2 tren tap Test
print(f"R2 Test  : {r2:.4f}")


MUC 5 - RANDOM FOREST HOLD-OUT
----------------------------------------
MSE Test : 271461.2833
RMSE Test: 521.0195
MAE Test : 263.9330
R2 Test  : 0.9828


So sánh mức 3, 4 và 5

In [8]:

# CELL 18: SO SANH KET QUA HOLD-OUT

# Tao bang so sanh ba muc thuc nghiem
comparison = pd.DataFrame({
    "Muc": ["Muc 3", "Muc 4", "Muc 5"],  # Ten cac muc
    "So cau hinh": [8, 36, 90],  # So cau hinh da thu
    "RMSE": [520.3564, 517.5292, rmse],  # RMSE Test
    "MAE": [264.5281, 263.2254, mae],  # MAE Test
    "R2": [0.9829, 0.9831, r2]  # R2 Test
})

# Hien thi bang ket qua, lam tron 4 chu so
display(comparison.round(4))

# Tinh chenh lech RMSE giua muc 5 va muc 4
rmse_difference = rmse - 517.5292

# Tinh phan tram thay doi RMSE so voi muc 4
percentage = rmse_difference / 517.5292 * 100

# In chenh lech RMSE
print(f"Chenh lech RMSE: {rmse_difference:+.4f}")

# In phan tram thay doi RMSE
print(f"Thay doi RMSE: {percentage:+.2f}%")

# Kiem tra muc 5 co RMSE thap hon muc 4 khong
if rmse_difference < 0:
    print("Muc 5 co RMSE thap hon Muc 4.")

# Kiem tra muc 5 co RMSE cao hon muc 4 khong
elif rmse_difference > 0:
    print("Muc 5 co RMSE cao hon Muc 4.")

# Truong hop hai muc co RMSE bang nhau
else:
    print("RMSE hai muc bang nhau.")


,Muc,So cau hinh,RMSE,MAE,R2
0,Muc 3,8,520.3564,264.5281,0.9829
1,Muc 4,36,517.5292,263.2254,0.9831
2,Muc 5,90,521.0195,263.9330,0.9828


Chenh lech RMSE: +3.4903
Thay doi RMSE: +0.67%
Muc 5 co RMSE cao hon Muc 4.


In [9]:

# CELL 19: HIEN THI 10 CAU HINH TOT NHAT

# Chuyen ket qua GridSearchCV thanh DataFrame
cv_results = pd.DataFrame(grid_search.cv_results_)

# Lay cac cot thong tin can thiet
top10 = cv_results[
    [
        "rank_test_score",  # Thu hang cua cau hinh
        "param_model__n_estimators",  # So cay
        "param_model__max_depth",  # Do sau toi da
        "param_model__min_samples_leaf",  # So mau toi thieu tai la
        "param_model__max_features",  # Ty le dac trung
        "mean_test_score",  # Diem CV trung binh
        "std_test_score"  # Do lech chuan diem CV
    ]
].copy()  # Tao ban sao de xu ly

# Chuyen diem RMSE dang am thanh RMSE duong
top10["RMSE_CV"] = -top10["mean_test_score"]

# Sap xep cau hinh theo RMSE CV tang dan
top10 = top10.sort_values("RMSE_CV")

# Chi lay 10 cau hinh co RMSE CV thap nhat
top10 = top10.head(10)

# Hien thi bang ket qua voi 4 chu so thap phan
display(top10.round(4))


,rank_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_test_score,std_test_score,RMSE_CV
64,1,300,30,1,0.6,-552.2089,2.6341,552.2089
63,2,250,30,1,0.6,-552.2556,2.1105,552.2556
19,3,300,None,2,0.8,-552.2769,2.7915,552.2769
79,4,300,30,2,0.8,-552.3376,2.7519,552.3376
77,5,200,30,2,0.8,-552.4976,2.3448,552.4976
17,6,200,None,2,0.8,-552.4976,2.3995,552.4976
78,7,250,30,2,0.8,-552.5777,2.5911,552.5777
18,8,250,None,2,0.8,-552.5827,2.6385,552.5827
38,9,250,20,2,0.6,-552.6357,2.7130,552.6357
4,10,300,None,1,0.6,-552.6617,2.4296,552.6617
